# 05 · Writers TMDL/PBIR và build Power BI Project

Notebook này nhận `model_spec.json` và `layout.json` của job đã duyệt, sinh TMDL cho semantic model, sinh PBIR từ visual templates xuất bằng Power BI Desktop, rồi đóng gói thành `.pbip`.

Luồng: `model_spec + data_catalog → TMDL` và `layout + PBIR templates → PBIR`; copy metadata chuẩn từ project mẫu; cập nhật `definition.pbir` và `.pbip`; validate file; mở Power BI Desktop → Refresh → kiểm tra từng visual.

**PBIR cần visual template có field binding thật.** Các visual mẫu hiện có trong repo chỉ khai báo `visualType`, chưa bind cột/measure; notebook sẽ từ chối dùng chúng để tránh tạo report trống. Hãy tạo template bằng Power BI Desktop một lần theo hướng dẫn trong cell cuối.

In [ ]:
from pathlib import Path
import json
import re
import shutil
import tempfile
import uuid

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy repo root.')

ROOT = find_repo_root()
RUNS_DIR = ROOT / 'runs'
JOB_ID = None  # Đặt job_id để chọn job cụ thể; None chọn job gần nhất đủ input.
INPUT_FILES = ('requirement.json', 'data_catalog.json', 'model_spec.json', 'layout.json')
if JOB_ID:
    RUN_DIR = RUNS_DIR / JOB_ID
else:
    eligible = [p for p in RUNS_DIR.glob('*') if all((p / name).is_file() for name in INPUT_FILES)]
    if not eligible:
        raise FileNotFoundError('Cần chạy notebook 01, 02, 03 và lưu đủ requirement/catalog/model/layout trước.')
    RUN_DIR = max(eligible, key=lambda p: p.stat().st_mtime)
JOB_ID = RUN_DIR.name
requirement = json.loads((RUN_DIR / 'requirement.json').read_text(encoding='utf-8'))
catalog = json.loads((RUN_DIR / 'data_catalog.json').read_text(encoding='utf-8'))
model_spec = json.loads((RUN_DIR / 'model_spec.json').read_text(encoding='utf-8'))
layout_doc = json.loads((RUN_DIR / 'layout.json').read_text(encoding='utf-8'))
layout = layout_doc.get('layout', layout_doc)
SOURCE_FILE = Path(catalog['source_file'])
SAMPLE_ROOT = ROOT / 'data' / 'data_samples'
REPORT_TEMPLATES = list(SAMPLE_ROOT.glob('*.Report'))
MODEL_TEMPLATES = list(SAMPLE_ROOT.glob('*.SemanticModel'))
if len(REPORT_TEMPLATES) != 1 or len(MODEL_TEMPLATES) != 1:
    raise FileNotFoundError('Cần đúng một cặp *.Report và *.SemanticModel trong data/data_samples/.')
REPORT_TEMPLATE, MODEL_TEMPLATE = REPORT_TEMPLATES[0], MODEL_TEMPLATES[0]
VISUAL_TEMPLATE_DIR = ROOT / 'templates' / 'pbir' / 'visuals'

def safe_project_name(value):
    result = re.sub(r'[^A-Za-z0-9_-]+', '_', str(value)).strip('_')
    return result or 'PBI_Report'

PROJECT_NAME = safe_project_name(requirement.get('requirement', {}).get('report_name', 'PBI_Report'))
REPORT_FOLDER = f'{PROJECT_NAME}.Report'
MODEL_FOLDER = f'{PROJECT_NAME}.SemanticModel'
OUTPUT_ROOT = RUN_DIR / 'output'
print('Job:', JOB_ID, '| Source:', SOURCE_FILE, '| Output:', OUTPUT_ROOT)
print('PBIR visual templates:', VISUAL_TEMPLATE_DIR)

## Writer 1 · Semantic model TMDL

Khung đầu ra: `definition/model.tmdl` chứa model, table refs và relationships; `definition/tables/<table>.tmdl` chứa cột, measure và M partition. Nguồn hiện hỗ trợ CSV, Excel, Parquet và JSON dạng mảng records. Cột có `source_column: null` phải có `expression` DAX; xử lý biến đổi Power Query phức tạp cần mở rộng `render_m_partition`.

In [ ]:
DATA_TYPES = {
    'string': ('string', 'type text'),
    'int64': ('int64', 'Int64.Type'),
    'double': ('double', 'type number'),
    'decimal': ('decimal', 'Currency.Type'),
    'dateTime': ('dateTime', 'type datetime'),
    'date': ('dateTime', 'type date'),
    'boolean': ('boolean', 'type logical'),
}

def tmdl_name(value):
    value = str(value)
    if re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', value):
        return value
    return "'" + value.replace("'", "''") + "'"

def m_text(value):
    return '"' + str(value).replace('"', '""') + '"'

def field_types(table):
    pairs = []
    for column in table.get('columns', []):
        source_column = column.get('source_column')
        if source_column:
            data_type = column.get('data_type', 'string')
            if data_type not in DATA_TYPES:
                raise ValueError(f"data_type không hỗ trợ: {data_type}")
            pairs.append((source_column, DATA_TYPES[data_type][1]))
    return pairs

def render_m_partition(table):
    suffix = SOURCE_FILE.suffix.lower()
    source_path = m_text(SOURCE_FILE.resolve())
    if suffix == '.csv':
        lines = [f'Source = Csv.Document(File.Contents({source_path}), [Delimiter=",", Encoding=65001, QuoteStyle=QuoteStyle.Csv]),',
                 'Promoted = Table.PromoteHeaders(Source, [PromoteAllScalars=true]),']
        previous = 'Promoted'
    elif suffix in {'.xlsx', '.xlsm'}:
        source_name = table.get('source_table')
        lines = [f'Source = Excel.Workbook(File.Contents({source_path}), null, true),',
                 f'Raw = Source{{[Item={m_text(source_name)}, Kind="Sheet"]}}[Data],',
                 'Promoted = Table.PromoteHeaders(Raw, [PromoteAllScalars=true]),']
        previous = 'Promoted'
    elif suffix == '.parquet':
        lines = [f'Source = Parquet.Document(File.Contents({source_path})),']
        previous = 'Source'
    elif suffix == '.json':
        with SOURCE_FILE.open(encoding='utf-8') as source_stream:
            json_shape = json.load(source_stream)
        if not isinstance(json_shape, list) or any(not isinstance(row, dict) for row in json_shape[:10]):
            raise ValueError('JSON chỉ hỗ trợ top-level array gồm object records; định dạng khác cần M query riêng.')
        lines = [f'Source = Json.Document(File.Contents({source_path})),',
                 'AsTable = Table.FromRecords(Source),']
        previous = 'AsTable'
    else:
        raise ValueError(f'Nguồn chưa hỗ trợ: {suffix}')
    type_pairs = field_types(table)
    if type_pairs:
        rendered_pairs = ', '.join('{' + m_text(name) + ', ' + m_type + '}' for name, m_type in type_pairs)
        lines.append(f'Typed = Table.TransformColumnTypes({previous}, {{{rendered_pairs}}}),')
        previous = 'Typed'
    source_columns = [column['source_column'] for column in table.get('columns', []) if column.get('source_column')]
    if source_columns:
        rendered_columns = ', '.join(m_text(name) for name in source_columns)
        lines.append(f'Selected = Table.SelectColumns({previous}, {{{rendered_columns}}}, MissingField.Ignore)')
        previous = 'Selected'
    return lines, previous

def render_table_tmdl(table):
    table_name = table['name']
    lines = [f'table {tmdl_name(table_name)}', f'\tlineageTag: {uuid.uuid4()}', '']
    direct_columns = [column for column in table.get('columns', []) if column.get('source_column')]
    if not direct_columns:
        raise ValueError(f"Table {table_name} cần ít nhất một cột nguồn để tạo import partition.")
    for column in table.get('columns', []):
        data_type = column.get('data_type', 'string')
        if data_type not in DATA_TYPES:
            raise ValueError(f"data_type không hỗ trợ: {data_type}")
        name = tmdl_name(column['name'])
        expression = column.get('expression')
        if column.get('source_column'):
            lines.extend([f'\tcolumn {name}', f'\t\tdataType: {DATA_TYPES[data_type][0]}',
                          f'\t\tsourceColumn: {tmdl_name(column["source_column"])}',
                          f'\t\tsummarizeBy: {"none" if column.get("role") in {"key", "dimension"} else "sum"}',
                          '\t\tannotation SummarizationSetBy = Automatic', ''])
        elif expression:
            lines.extend([f'\tcolumn {name} = {expression}', f'\t\tdataType: {DATA_TYPES[data_type][0]}',
                          '\t\tsummarizeBy: none', ''])
        else:
            raise ValueError(f"Cột {column['name']} cần source_column hoặc expression DAX.")
    table_measures = {measure['name']: measure for measure in table.get('measures', [])}
    for measure in model_spec.get('measures', []):
        if measure.get('table') == table_name:
            if measure['name'] in table_measures:
                raise ValueError(f"Measure bị khai báo lặp trong model_spec: {measure['name']}")
            table_measures[measure['name']] = measure
    for measure in table_measures.values():
        lines.extend([f'\tmeasure {tmdl_name(measure["name"])} = {measure["expression"]}',
                      f'\t\tformatString: {measure.get("format_string", "General")}', ''])
    m_lines, last_step = render_m_partition(table)
    lines.extend([f'\tpartition {tmdl_name(table_name)} = m', '\t\tmode: import', '\t\tsource ='])
    lines.extend('\t\t\t' + line for line in m_lines)
    lines.extend(['\t\tin', '\t\t\t' + last_step, '', '\tannotation PBI_ResultType = Table', ''])
    return '\n'.join(lines)

def render_model_tmdl(spec):
    names = [table['name'] for table in spec.get('tables', [])]
    if not names or len(set(names)) != len(names):
        raise ValueError('model_spec cần table name duy nhất và ít nhất một table.')
    lines = ['model Model', '\tculture: en-US', '\tdefaultPowerBIDataSourceVersion: powerBI_V3',
             '\tsourceQueryCulture: en-US', '\tdataAccessOptions', '\t\tlegacyRedirects',
             '\t\treturnErrorValuesAsNull', '',
             '\tannotation PBI_QueryOrder = ' + json.dumps(names, ensure_ascii=False), '',
             '\tannotation __PBI_TimeIntelligenceEnabled = 1', '']
    lines.extend(f'\tref table {tmdl_name(name)}' for name in names)
    lines.extend(['\tref cultureInfo en-US', ''])
    for index, relationship in enumerate(spec.get('relationships', []), start=1):
        cardinality = relationship.get('cardinality', 'manyToOne')
        if cardinality not in {'manyToOne', 'oneToMany', 'oneToOne', 'manyToMany'}:
            raise ValueError(f'Cardinality không hỗ trợ: {cardinality}')
        from_card, to_card = {'manyToOne': ('many', 'one'), 'oneToMany': ('one', 'many'),
                             'oneToOne': ('one', 'one'), 'manyToMany': ('many', 'many')}[cardinality]
        rel_name = f'Relationship_{index}_{relationship["from_table"]}_{relationship["to_table"]}'
        lines.extend([f'\trelationship {tmdl_name(rel_name)}',
                      f'\t\tfromColumn: {tmdl_name(relationship["from_table"])}.{tmdl_name(relationship["from_column"])}',
                      f'\t\ttoColumn: {tmdl_name(relationship["to_table"])}.{tmdl_name(relationship["to_column"])}',
                      f'\t\tfromCardinality: {from_card}', f'\t\ttoCardinality: {to_card}',
                      f'\t\tcrossFilteringBehavior: {"bothDirections" if relationship.get("cross_filtering") == "both" else "oneDirection"}',
                      f'\t\tisActive: {str(bool(relationship.get("is_active", True))).lower()}', ''])
    return '\n'.join(lines)

print('Writer TMDL sẵn sàng cho', len(model_spec.get('tables', [])), 'table(s).')

## Writer 2 · Report PBIR

PBIR là bộ file JSON (`definition/report.json`, `pages/pages.json`, `page.json`, `visual.json`) kèm resource/theme. Page và visual container được ghi theo layout; phần `visual` query giữ cấu trúc do Power BI Desktop sinh. Chỉ thay marker field trong template, không tự chế schema binding.

Quy ước template: `templates/pbir/visuals/<visualType>__<role1>__<role2>.json`; mỗi visual template phải có các chuỗi `__FIELD_0_TABLE__`, `__FIELD_0_NAME__`, ... đúng thứ tự fields trong layout. Layout có thể đặt `template_key` để dùng tên khác.

In [ ]:
VISUAL_TYPES = {'bar': 'barChart', 'column': 'clusteredColumnChart', 'line': 'lineChart',
                'donut': 'donutChart', 'card': 'cardVisual', 'table': 'tableEx', 'slicer': 'slicer'}

def model_field_map(spec):
    result = {}
    for table in spec.get('tables', []):
        for column in table.get('columns', []):
            result[f"{table['name']}[{column['name']}]"] = (table['name'], column['name'])
        for measure in table.get('measures', []):
            result[f"{table['name']}[{measure['name']}]"] = (table['name'], measure['name'])
    for measure in spec.get('measures', []):
        result[f"{measure['table']}[{measure['name']}]"] = (measure['table'], measure['name'])
    return result

FIELD_MAP = model_field_map(model_spec)

def visual_template_key(visual):
    if visual.get('template_key'):
        return visual['template_key']
    raw_type = visual.get('type', '')
    visual_type = VISUAL_TYPES.get(raw_type, raw_type)
    roles = [re.sub(r'[^a-z0-9]+', '_', field.get('role', 'value').lower()).strip('_')
             for field in visual.get('fields', [])]
    return '__'.join([visual_type, *roles])

def normalized_field(field):
    value = field.get('field', '')
    if value not in FIELD_MAP:
        raise ValueError(f'Field layout không tồn tại trong model_spec: {value}')
    return FIELD_MAP[value]

def expected_visual_templates(layout_spec):
    expected = {}
    for page in layout_spec.get('pages', []):
        for visual in page.get('visuals', []):
            key = visual_template_key(visual)
            expected[key] = visual
            for field in visual.get('fields', []):
                normalized_field(field)
    return expected

def replace_tokens(value, replacements):
    if isinstance(value, str):
        for token, replacement in replacements.items():
            value = value.replace(token, replacement)
        return value
    if isinstance(value, list):
        return [replace_tokens(item, replacements) for item in value]
    if isinstance(value, dict):
        return {key: replace_tokens(item, replacements) for key, item in value.items()}
    return value

def render_visual(template_dir, visual, page_size):
    key = visual_template_key(visual)
    template_path = template_dir / f'{key}.json'
    if not template_path.is_file():
        raise FileNotFoundError(f'Thiếu PBIR template {template_path}. Tạo visual trong Power BI Desktop, export PBIR, đặt tên theo template_key.')
    template_text = template_path.read_text(encoding='utf-8')
    template = json.loads(template_text)
    replacements = {}
    for index, field in enumerate(visual.get('fields', [])):
        table_name, field_name = normalized_field(field)
        replacements[f'__FIELD_{index}_TABLE__'] = table_name
        replacements[f'__FIELD_{index}_NAME__'] = field_name
    if '__VISUAL_TITLE__' in template_text:
        replacements['__VISUAL_TITLE__'] = visual.get('title', '')
    missing_tokens = [token for token in replacements if token not in template_text]
    if missing_tokens:
        raise ValueError(f'Template {template_path.name} thiếu marker: {missing_tokens}')
    rendered = replace_tokens(template, replacements)
    position = visual.get('position', {})
    x, y, width, height = (float(position.get(key, default)) for key, default in
                           [('x', 0), ('y', 0), ('width', 1), ('height', 0.25)])
    scale_x, scale_y = page_size['width'], page_size['height']
    rendered['name'] = uuid.uuid4().hex[:20]
    rendered['position'] = {'x': round(x * scale_x), 'y': round(y * scale_y), 'z': 0,
                            'width': round(width * scale_x), 'height': round(height * scale_y),
                            'tabOrder': 0}
    return rendered

def render_pbir_pages(layout_spec, template_dir):
    pages_dir = {}
    page_order = []
    for page in layout_spec.get('pages', []):
        page_id = uuid.uuid4().hex[:20]
        page_order.append(page_id)
        page_size = {'width': int(page.get('width', 1280)), 'height': int(page.get('height', 720))}
        visuals = {}
        for visual in page.get('visuals', []):
            rendered = render_visual(template_dir, visual, page_size)
            visuals[rendered['name']] = rendered
        page_json = {'$schema': 'https://developer.microsoft.com/json-schemas/fabric/item/report/definition/page/2.1.0/schema.json',
                     'name': page_id, 'displayName': page.get('name', 'Page'),
                     'displayOption': 'FitToPage', **page_size}
        pages_dir[page_id] = {'page.json': page_json, 'visuals': visuals}
    if not page_order:
        raise ValueError('layout.json cần ít nhất một page.')
    pages_index = {'$schema': 'https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.1.0/schema.json',
                   'pageOrder': page_order, 'activePageName': page_order[0]}
    return pages_index, pages_dir

print('Writer PBIR sẵn sàng. Templates cần thiết:', sorted(expected_visual_templates(layout)))

## Build · Tạo `.pbip`

Cell cuối kiểm tra toàn bộ input/template trước khi ghi; dựng trong thư mục tạm; chỉ chuyển sang `runs/<job>/output/` sau khi TMDL/PBIR JSON hợp lệ. Không ghi đè output có sẵn.

In [ ]:
def validate_writer_inputs():
    if not SOURCE_FILE.is_file():
        raise FileNotFoundError(f'Không tìm thấy data source: {SOURCE_FILE}')
    if not model_spec.get('tables'):
        raise ValueError('model_spec.json chưa có tables.')
    known_catalog = {table['name']: {column['name'] for column in table['columns']}
                     for table in catalog.get('tables', [])}
    model_names = {table['name'] for table in model_spec['tables']}
    for table in model_spec['tables']:
        source_name = table.get('source_table')
        if source_name not in known_catalog:
            raise ValueError(f"source_table không có trong catalog: {source_name}")
        for column in table.get('columns', []):
            source_column = column.get('source_column')
            if source_column and source_column not in known_catalog[source_name]:
                raise ValueError(f"Cột nguồn không có trong catalog: {source_name}[{source_column}]")
    for relationship in model_spec.get('relationships', []):
        if relationship.get('from_table') not in model_names or relationship.get('to_table') not in model_names:
            raise ValueError(f'Relationship tham chiếu table không tồn tại: {relationship}')
    templates = expected_visual_templates(layout)
    if not templates:
        raise ValueError('layout.json chưa có visual nào.')
    missing = [key for key in templates if not (VISUAL_TEMPLATE_DIR / f'{key}.json').is_file()]
    if missing:
        raise FileNotFoundError('Thiếu PBIR visual template: ' + ', '.join(missing) +
                              '. Hãy export visual đã bind field từ Power BI Desktop theo hướng dẫn cuối notebook.')
    if OUTPUT_ROOT.exists():
        raise FileExistsError(f'Output đã tồn tại: {OUTPUT_ROOT}. Đổi job_id hoặc di chuyển output cũ trước khi chạy lại.')

def build_powerbi_project():
    validate_writer_inputs()
    generated_tables = {}
    for table in model_spec['tables']:
        filename = re.sub(r'[^A-Za-z0-9_-]+', '_', table['name']).strip('_') or 'Table'
        if filename + '.tmdl' in generated_tables:
            raise ValueError(f'Nama file table bentrok setelah sanitasi: {filename}.tmdl')
        generated_tables[filename + '.tmdl'] = render_table_tmdl(table)
    model_tmdl = render_model_tmdl(model_spec)
    pages_index, pages = render_pbir_pages(layout, VISUAL_TEMPLATE_DIR)
    if len(REPORT_TEMPLATES) != 1 or len(MODEL_TEMPLATES) != 1:
        raise FileNotFoundError('Report/SemanticModel template tidak lengkap.')
    with tempfile.TemporaryDirectory(prefix='pbi_writer_', dir=RUN_DIR) as temp_name:
        stage = Path(temp_name) / 'output'
        stage.mkdir()
        report_root, model_root = stage / REPORT_FOLDER, stage / MODEL_FOLDER
        shutil.copytree(REPORT_TEMPLATE, report_root, ignore=shutil.ignore_patterns('.pbi'))
        shutil.copytree(MODEL_TEMPLATE, model_root, ignore=shutil.ignore_patterns('.pbi'))
        definition = model_root / 'definition'
        tables_dir = definition / 'tables'
        tables_dir.mkdir(exist_ok=True)
        for old_table in tables_dir.glob('*.tmdl'):
            old_table.unlink()
        (definition / 'model.tmdl').write_text(model_tmdl, encoding='utf-8')
        for filename, contents in generated_tables.items():
            (tables_dir / filename).write_text(contents, encoding='utf-8')
        diagram = model_root / 'diagramLayout.json'
        if diagram.exists():
            diagram.unlink()
        report_definition = report_root / 'definition'
        pages_root = report_definition / 'pages'
        if pages_root.exists():
            shutil.rmtree(pages_root)
        pages_root.mkdir(parents=True)
        (pages_root / 'pages.json').write_text(json.dumps(pages_index, ensure_ascii=False, indent=2), encoding='utf-8')
        for page_id, page in pages.items():
            page_root = pages_root / page_id
            page_root.mkdir()
            (page_root / 'page.json').write_text(json.dumps(page['page.json'], ensure_ascii=False, indent=2), encoding='utf-8')
            visuals_root = page_root / 'visuals'
            for visual_id, visual_json in page['visuals'].items():
                visual_dir = visuals_root / visual_id
                visual_dir.mkdir(parents=True)
                (visual_dir / 'visual.json').write_text(json.dumps(visual_json, ensure_ascii=False, indent=2), encoding='utf-8')
        pbir_path = report_root / 'definition.pbir'
        pbir = json.loads(pbir_path.read_text(encoding='utf-8'))
        pbir.setdefault('datasetReference', {}).setdefault('byPath', {})['path'] = f'../{MODEL_FOLDER}'
        pbir_path.write_text(json.dumps(pbir, ensure_ascii=False, indent=2), encoding='utf-8')
        descriptor = {'version': '1.0', 'artifacts': {'report': {'path': REPORT_FOLDER},
                       'semanticModel': {'path': MODEL_FOLDER}},
                       'settings': {'enableAutoRecovery': True}}
        (stage / f'{PROJECT_NAME}.pbip').write_text(json.dumps(descriptor, ensure_ascii=False, indent=2), encoding='utf-8')
        for json_path in report_root.rglob('*.json'):
            json.loads(json_path.read_text(encoding='utf-8'))
        json.loads((stage / f'{PROJECT_NAME}.pbip').read_text(encoding='utf-8'))
        OUTPUT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        stage.rename(OUTPUT_ROOT)
    return OUTPUT_ROOT / f'{PROJECT_NAME}.pbip'

# Chạy sau khi đã tạo mọi visual template PBIR cần thiết.
# pbip_path = build_powerbi_project()
# print('Mở bằng Power BI Desktop:', pbip_path)

## Tạo PBIR visual templates (làm một lần cho mỗi kiểu/role)

1. Trong Power BI Desktop, tạo report nền có một bảng tên `__FIELD_0_TABLE__` và cột/measure tên `__FIELD_0_NAME__`, tiếp tục đánh số cho từng field của visual.
2. Tạo visual đúng loại và đúng roles (ví dụ bar chart: category rồi value), lưu project dạng PBIP/PBIR.
3. Copy file `visual.json` của visual đó vào `templates/pbir/visuals/` với tên như `barChart__category__value.json`. Các chuỗi marker phải còn nguyên trong file; có thể dùng `__VISUAL_TITLE__` làm marker tiêu đề.
4. Tạo template riêng cho mỗi tổ hợp visual type/roles. Sau đó chạy notebook từ đầu, kiểm tra model/layout, rồi chạy `build_powerbi_project()`.

Output gồm `<name>.pbip`, `<name>.Report/` (PBIR) và `<name>.SemanticModel/` (TMDL). Mở file `.pbip` trong Power BI Desktop, kiểm tra credentials/path, Refresh, xác nhận quan hệ/DAX và mọi visual trước khi bàn giao. Kiểm tra JSON trong notebook không thay thế validate/refresh của Desktop.